# Chapter 8 &mdash; Regular Expressions Are Error-Prone &mdash; a Security Problem

**Concept 4 of the Chapter 8 decomposition:** *Regular Expressions Are Error-Prone, and That Is a Security Problem*

Real-world RE features and an unanchored `\d+-\d+` check that a crafted input walks straight through.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-RE-Are-Error-Prone/Concept-RE-Are-Error-Prone.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Real-world regular expressions are not the clean algebra above. They add anchors
(`^`, `$`), character classes (`\d`), backreferences, and lazy quantifiers &mdash; and the
extra features make them **easy to get subtly wrong**.

The classic failure is a **missing anchor**. A validator that tests `\d+-\d+`
*anywhere in the string* accepts `abc 12-34; rm -rf /` because the pattern **does**
occur &mdash; just not alone. In a language that interpolates the whole input into a
shell command or a query, that is an injection.

The lesson generalises: **an RE that matches is not an RE that validates.** Anchor it,
and test the negative cases.

## 2. Definitions

### The careless validator and the careful one

In [ ]:
import re as pyre
careless = pyre.compile(r'\d+-\d+')            # matches ANYWHERE
careful  = pyre.compile(r'^\d+-\d+$')          # anchored at both ends

def check(rx, s): return rx.search(s) is not None

### The same language as a clean RE, over a small alphabet

In [ ]:
def re_dfa(r): return min_dfa(nfa2dfa(re2nfa(r)))
# digits abbreviated to {0,1} and '-' to 'd' so Jove's single-char alphabet fits
clean = "(0+1)(0+1)*d(0+1)(0+1)*"

## 3. Tests

Both accept a legitimate range.

In [ ]:
for s in ['12-34', '1-2', '007-042']:
    print("%-10r careless=%-5s careful=%s" % (s, check(careless, s), check(careful, s)))
assert check(careless, '12-34') and check(careful, '12-34')

**The bug:** the unanchored pattern accepts an input with a payload attached.

In [ ]:
attacks = ['abc 12-34', '12-34; rm -rf /', '../../etc 1-2', '12-34\nDROP TABLE t']
for s in attacks:
    print("%-24r careless=%-6s careful=%s"
          % (s, check(careless, s), check(careful, s)))
assert all(check(careless, s) for s in attacks)
assert not any(check(careful, s) for s in attacks)
print("\nEvery attack string passes the careless check and fails the careful one.")

The anchored version is the one that corresponds to the **language** $\{d^+\text{-}d^+\}$.

In [ ]:
D = re_dfa(clean)
for s in ['1d1', '10d01', 'd1', '1d', '1d1d1', '']:
    print("%-8r in the clean language? %s" % (s, accepts_dfa(D, s)))
assert accepts_dfa(D, '10d01') and not accepts_dfa(D, '1d1d1')
print("\nAn automaton has no notion of 'matches somewhere' -- it either accepts the")
print("WHOLE string or it does not.  That is exactly what anchoring restores.")

Anchoring, expressed automata-theoretically: unanchored search is $\Sigma^* R \Sigma^*$.

In [ ]:
unanchored = "(0+1+d)*" + "(" + clean + ")" + "(0+1+d)*"
U = re_dfa(unanchored)
print("unanchored accepts '1d1d1'? ", accepts_dfa(U, '1d1d1'))
print("anchored   accepts '1d1d1'? ", accepts_dfa(D, '1d1d1'))
assert accepts_dfa(U, '1d1d1') and not accepts_dfa(D, '1d1d1')
print("\nThe two Sigma* wrappers ARE the missing anchors, made visible.")

## 4. Animation

The anchored machine &mdash; it accepts only a whole well-formed range.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(re_dfa(clean), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Write the anchored RE for an IPv4 address. How many negative tests did you need?
2. What is "catastrophic backtracking", and why can a DFA never suffer from it?
3. Which real-world RE features take the language *outside* the regular class?

In [ ]:
# Your work for the exercises above.